In [1]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time sm7cUdHWy1vP5b3lsEANUEIn

crunch-cli, version 11.10.0
main.py: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/69432/main.py (17454 bytes)
notebook.ipynb: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/69432/notebook.ipynb (42137 bytes)
requirements.txt: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/69432/requirements.txt (190 bytes)
resources/model.joblib: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/models/62431/model.joblib (4001 bytes)
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/y_train.parquet: download from https:crunchdao--competition--producti

In [2]:
import math
import os
from typing import Iterable, List, Optional, Tuple

# Import your dependencies.
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [3]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

loaded crunch tools for module: <module '__main__'>

cli version: 11.10.0
available ram: 12.67 gb
available cpu: 2 core
----


## Looking at the data

`crunch_tools.load_data()` returns two objects:

- `train_data`: the training set. <br />
  Each element is a tuple `(dataset_id, x_historical, x_online, tau_index)` where:
  - `dataset_id` is an integer identifier,
  - `x_historical` is a 1-D numpy array of the historical segment,
  - `x_online` is a 1-D numpy array of the full online segment,
  - `tau_index` is the index within `x_online` at which the break happens, or `None` if the series has no break.
- `test_data`: the test set. <br />
  with the same layout but *without* `tau_index`, and with `x_online` that can only be **iterated once** in the cloud environment.

<br />

> **📝 Note**
> ---
> Locally (inside this notebook) you can re-read the test series as many times as you like while developing.

In [4]:
# Load the data.
train_data, test_data = crunch_tools.load_data()

data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

In [5]:
from collections import deque
import math
import numpy as np

def _win_stats_blocks(zh, w):
    n = len(zh)
    if n < 2*w:
        return None
    step = max(w // 2, 1)
    lvars, kss, skews, ac1s, means = [], [], [], [], []
    qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
    for start in range(0, n - w + 1, step):
        seg = zh[start:start+w]
        m = seg.mean(); v = max(seg.var(), 1e-12)
        lvars.append(math.log(v))
        zn = (seg - m)/math.sqrt(v)
        skews.append(float((zn**3).mean()))
        a = seg - m
        ac1s.append(float(np.dot(a[1:], a[:-1]) / max(np.dot(a, a), 1e-12)))
        emp = np.searchsorted(np.sort(seg), qs, side="right") / w
        kss.append(float(np.max(np.abs(emp - np.linspace(0.1, 0.9, 9)))))
        means.append(m)
    def musd(v):
        v = np.asarray(v); return float(v.mean()), max(float(v.std()), 1e-6)
    return {"lvar": musd(lvars), "ks": musd(kss), "skew": musd(skews),
            "ac1": musd(ac1s), "mean": musd(means)}

def _fit_ar2(z):
    if len(z) < 50:
        return 0.0, 0.0, 1.0
    y = z[2:]; X = np.column_stack([z[1:-1], z[:-2]])
    XtX = X.T @ X + 1e-6 * np.eye(2)
    phi = np.linalg.solve(XtX, X.T @ y)
    resid = y - X @ phi
    return float(phi[0]), float(phi[1]), max(float(resid.var()), 1e-6)

class StreamingFeatures:
    def __init__(self, x_hist):
        x = np.asarray(x_hist, dtype=np.float64)
        self.mu_h = float(x.mean()); self.sd_h = max(float(x.std(ddof=1)), 1e-8)
        xc = x - self.mu_h
        den_h = max(np.dot(xc, xc), 1e-12)
        self.ac1_h = float(np.dot(xc[1:], xc[:-1]) / den_h)
        self.ac2_h = float(np.dot(xc[2:], xc[:-2]) / den_h)
        self.ac5_h = float(np.dot(xc[5:], xc[:-5]) / den_h)
        zh = xc / self.sd_h
        self.qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
        self.skew_h = float((zh**3).mean())
        self.kurt_h = float((zh**4).mean()) - 3.0
        self.cal100 = _win_stats_blocks(zh, 100)
        self.cal20 = _win_stats_blocks(zh, 20)
        self.t = 0; self.sum = 0.0; self.sumsq = 0.0
        self.cusum_pos = 0.0; self.cusum_neg = 0.0; self.n_tail = 0
        self.prev_z = None; self.ac1_num = 0.0; self.ac1_den = 0.0
        self.w20 = deque(); self.s20 = 0.0; self.q20 = 0.0
        self.w100 = deque(maxlen=100); self.s100 = 0.0; self.q100 = 0.0
        self.pz = [0.0]; self.pz2 = [0.0]; self.pz3 = [0.0]; self.pt = [0]
        self.phi1, self.phi2, self.rv = _fit_ar2(zh)
        self.z1 = zh[-1]; self.z2 = zh[-2] if len(zh) >= 2 else 0.0
        self.ll_cum = 0.0
        self.r_sum = 0.0; self.r_sq = 0.0; self.r_n = 0
        self.r_prev = None; self.r_ac_num = 0.0; self.r_ac_den = 0.0
        self.rw = deque(maxlen=50); self.rw_s = 0.0; self.rw_q = 0.0
        self.r_cusum = 0.0
        self.ew_var = 1.0; self.g_cusum = 0.0; self.g_ll = 0.0; self.g_n = 0
        self.u3 = 0.0; self.u4 = 3.0

    def update(self, x):
        self.t += 1
        z = (x - self.mu_h) / self.sd_h
        k = 0.5
        self.cusum_pos = max(0.0, self.cusum_pos + z - k)
        self.cusum_neg = max(0.0, self.cusum_neg - z - k)
        cusum = max(self.cusum_pos, self.cusum_neg)
        self.sum += x; self.sumsq += x * x
        if self.t > 1:
            m = self.sum / self.t
            var_o = max((self.sumsq - self.t * m * m) / (self.t - 1), 1e-12) / self.sd_h ** 2
        else:
            var_o = 1.0
        var_ratio = math.log(var_o)
        if abs(z) > 2.0: self.n_tail += 1
        tail_frac = self.n_tail / self.t
        if self.prev_z is not None:
            self.ac1_num += z * self.prev_z; self.ac1_den += z * z
        ac1_o = self.ac1_num / self.ac1_den if self.ac1_den > 1e-12 else 0.0
        self.prev_z = z

        self.w20.append(z); self.s20 += z; self.q20 += z * z
        if len(self.w20) > 20:
            old = self.w20.popleft(); self.s20 -= old; self.q20 -= old * old
        n20 = len(self.w20); m20 = self.s20 / n20
        v20 = max(self.q20 / n20 - m20 * m20, 1e-12)

        if len(self.w100) == 100:
            old = self.w100[0]; self.s100 -= old; self.q100 -= old * old
        self.w100.append(z); self.s100 += z; self.q100 += z * z
        n100 = len(self.w100); m100 = self.s100 / n100
        v100 = max(self.q100 / n100 - m100 * m100, 1e-12)
        arr = np.asarray(self.w100)
        zn = (arr - m100) / math.sqrt(v100)
        skew_raw = float((zn**3).mean())
        w100_skew = skew_raw - self.skew_h
        w100_kurt = float((zn**4).mean()) - 3.0 - self.kurt_h
        a = arr - m100
        den = max(np.dot(a, a), 1e-12)
        ac1_100 = float(np.dot(a[1:], a[:-1]) / den) if n100 >= 5 else self.ac1_h
        w100_ac1 = ac1_100 - self.ac1_h
        w100_ac2 = (float(np.dot(a[2:], a[:-2]) / den) if n100 >= 7 else self.ac2_h) - self.ac2_h
        w100_ac5 = (float(np.dot(a[5:], a[:-5]) / den) if n100 >= 12 else self.ac5_h) - self.ac5_h
        emp = np.searchsorted(np.sort(arr), self.qs, side="right") / n100
        ks100 = float(np.max(np.abs(emp - np.linspace(0.1, 0.9, 9))))
        signs = np.sign(arr)
        flips = float(np.mean(signs[1:] * signs[:-1] < 0)) if n100 >= 3 else 0.5

        c = self.cal100
        if c is not None and n100 >= 50:
            cal_lvar = (math.log(v100) - c["lvar"][0]) / c["lvar"][1]
            cal_ks   = (ks100 - c["ks"][0]) / c["ks"][1]
            cal_skew = (skew_raw - c["skew"][0]) / c["skew"][1]
            cal_ac1  = (ac1_100 - c["ac1"][0]) / c["ac1"][1]
        else:
            cal_lvar = cal_ks = cal_skew = cal_ac1 = 0.0
        c2 = self.cal20
        if c2 is not None and n20 >= 10:
            cal_mean20 = (m20 - c2["mean"][0]) / c2["mean"][1]
        else:
            cal_mean20 = 0.0

        self.pz.append(self.pz[-1] + z)
        self.pz2.append(self.pz2[-1] + z * z)
        self.pz3.append(self.pz3[-1] + z ** 3)
        self.pt.append(self.pt[-1] + (1 if abs(z) > 2 else 0))
        t = self.t
        scan_mean = scan_lvar = scan_skew = scan_tail = 0.0
        if t >= 8:
            for f in (0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85):
                s = max(2, int(t * f))
                if s >= t - 1: continue
                n1, n2 = s, t - s
                m1 = self.pz[s] / n1; m2 = (self.pz[t] - self.pz[s]) / n2
                v1 = max(self.pz2[s] / n1 - m1 * m1, 1e-12)
                v2 = max((self.pz2[t] - self.pz2[s]) / n2 - m2 * m2, 1e-12)
                dmean = abs(m2 - m1) / math.sqrt(v1 / n1 + v2 / n2)
                dlvar = abs(math.log(v2 / v1))
                mu3_1 = self.pz3[s] / n1 - 3 * m1 * v1 - m1 ** 3
                mu3_2 = (self.pz3[t] - self.pz3[s]) / n2 - 3 * m2 * v2 - m2 ** 3
                dskew = abs(mu3_2 / v2 ** 1.5 - mu3_1 / v1 ** 1.5)
                dtail = abs((self.pt[t] - self.pt[s]) / n2 - self.pt[s] / n1)
                if dmean > scan_mean: scan_mean = dmean
                if dlvar > scan_lvar: scan_lvar = dlvar
                if dskew > scan_skew: scan_skew = dskew
                if dtail > scan_tail: scan_tail = dtail

        pred = self.phi1 * self.z1 + self.phi2 * self.z2
        u = (z - pred) / math.sqrt(self.rv)
        self.z2 = self.z1; self.z1 = z
        self.ll_cum += 0.5 * (u * u - 1.0)
        self.r_n += 1; self.r_sum += u; self.r_sq += u * u
        r_var = self.r_sq / self.r_n - (self.r_sum / self.r_n) ** 2 if self.r_n > 1 else 1.0
        if self.r_prev is not None:
            self.r_ac_num += u * self.r_prev; self.r_ac_den += u * u
        r_ac = self.r_ac_num / self.r_ac_den if self.r_ac_den > 1e-12 else 0.0
        self.r_prev = u
        if len(self.rw) == 50:
            old = self.rw[0]; self.rw_s -= old; self.rw_q -= old * old
        self.rw.append(u); self.rw_s += u; self.rw_q += u * u
        nw = len(self.rw); mw = self.rw_s / nw
        vw = max(self.rw_q / nw - mw * mw, 1e-12)
        self.r_cusum = max(0.0, self.r_cusum + u - 0.5)
        ll_norm = self.ll_cum / math.sqrt(max(self.r_n, 1))

        g = u / math.sqrt(max(self.ew_var, 1e-6))
        self.ew_var = 0.94 * self.ew_var + 0.06 * u * u
        self.g_n += 1
        self.g_ll += 0.5 * (g * g - 1.0)
        self.g_cusum = max(0.0, self.g_cusum + abs(g) - 0.8)
        self.u3 = 0.97 * self.u3 + 0.03 * (u ** 3)
        self.u4 = 0.97 * self.u4 + 0.03 * (u ** 4)
        g_ll_norm = self.g_ll / math.sqrt(max(self.g_n, 1))
        g_cusum_n = self.g_cusum / math.sqrt(max(self.g_n, 1))

        return (self.t, cusum, var_ratio, tail_frac, ac1_o - self.ac1_h, abs(z),
                m20 * math.sqrt(n20), math.log(v20),
                m100 * math.sqrt(n100), math.log(v100),
                w100_skew, w100_kurt, w100_ac1, ks100, flips,
                cal_lvar, cal_ks, cal_skew, cal_ac1, cal_mean20,
                w100_ac2, w100_ac5, scan_mean, scan_lvar, scan_skew, scan_tail,
                ll_norm, math.log(max(r_var, 1e-12)), r_ac,
                math.log(vw), mw * math.sqrt(nw), self.r_cusum,
                g_ll_norm, g_cusum_n, math.log(max(self.ew_var, 1e-6)),
                self.u3, self.u4 - 3.0)

### The `train()` function

`train()` is called once before any prediction is made. <br />
Its job is to fit whatever model you want and save it to `model_directory_path`.

The baseline does not need training -- the EWMA z-score is computed entirely from the historical segment and the streaming online values, so there is no model to fit. <br />
We simply save a placeholder so `infer()` has something to load.

If you later decide to use a supervized learner (e.g. a gradient boosting model that takes features of the running state and predicts the break probability), this is the function where you would fit and save it.

In [6]:
%pip install catboost --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [7]:
import numpy as np
from catboost import CatBoostClassifier
from scipy.stats import norm as _norm
import gc, math

# @crunch/keep:on
INFER_PARALLELISM = 8

_LOOKUP_SIZE = 100_001
_LOOKUP_G = _norm.ppf(
    np.linspace(1.0 / _LOOKUP_SIZE, 1.0 - 1.0 / _LOOKUP_SIZE, _LOOKUP_SIZE)
).astype(np.float64)

def _rank_to_gauss(rank, n):
    p = rank / (n + 1)
    if p < 1.0 / (n + 1): p = 1.0 / (n + 1)
    elif p > n / (n + 1): p = n / (n + 1)
    return float(_LOOKUP_G[int(p * (_LOOKUP_SIZE - 1))])

def _rank_history(h_sorted, values):
    n = len(h_sorted)
    ranks = np.searchsorted(h_sorted, values, side="right")
    out = np.empty(len(values), dtype=np.float64)
    for i in range(len(values)):
        out[i] = _rank_to_gauss(int(ranks[i]), n)
    return out

def train(datasets, model_directory_path):
    import os, time

    MODEL_VERSION = "dualcat-full-1seed-v1"
    mpath = os.path.join(model_directory_path, "model.joblib")
    if os.path.exists(mpath):
        try:
            existing = joblib.load(mpath)
            if existing.get("version") == MODEL_VERSION:
                print(f"pre-trained model found ({MODEL_VERSION}) — skipping training")
                return
        except Exception as e:
            print(f"existing model unreadable ({e}) — retraining")

    datasets = list(datasets)
    t0 = time.time()

    rows, labels = [], []
    for k, (dataset_id, x_hist, x_online, tau) in enumerate(datasets):
        h = np.asarray(x_hist, dtype=np.float64)
        o = np.asarray(x_online, dtype=np.float64)
        h_sorted = np.sort(h)
        sf_raw = StreamingFeatures(h)
        sf_rank = StreamingFeatures(_rank_history(h_sorted, h))
        o_ranked = _rank_history(h_sorted, o)
        for t in range(len(o)):
            f_raw = sf_raw.update(float(o[t]))
            f_rank = sf_rank.update(float(o_ranked[t]))
            rows.append(f_raw + f_rank)
            labels.append(1 if (tau is not None and t >= tau) else 0)
        if (k + 1) % 2000 == 0:
            print(f"  features: {k+1}/{len(datasets)} series ({(time.time()-t0)/60:.0f} min)")

    X = np.asarray(rows, np.float32); del rows
    y = np.asarray(labels, np.int8); del labels
    gc.collect()
    print(f"features built: {X.shape} ({(time.time()-t0)/60:.0f} min)")

    cat = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                             random_seed=42, verbose=0, allow_writing_files=False,
                             thread_count=8)
    cat.fit(X, y)
    print(f"model fitted ({(time.time()-t0)/60:.0f} min)")

    del X, y; gc.collect()

    joblib.dump({"cat": cat, "version": MODEL_VERSION},
                os.path.join(model_directory_path, "model.joblib"))
    print(f"model saved ({(time.time()-t0)/60:.0f} min total)")

### The `infer()` function

`infer()` is a **generator**. It must:

1. Load any model saved by `train()`.
2. `yield` once, with no value, to signal readiness to the runner.
3. For each test series `(x_historical, x_online)`:
    - Pre-compute whatever static summaries you need from `x_historical` (it is given in full and cheap to scan once).
    - Loop over the points of `x_online`. After each point, `yield` a `float` in $[0, 1]$ -- **exactly one yield per online point**, in order.

<br />

> **‼ Important**
> ---
> Both the outer `datasets` iterable and each `x_online` can be iterated **only once** in the cloud environment. <br />
> You must produce the score for the current observation before the next one is released.

### The streaming EWMA detector in code

We keep three running scalars per series:

- `mu_ewma`: the EWMA of the online values (tracks the current local mean).
- `n_eff`: the effective sample size of the EWMA (grows as more points arrive, bounded by `1 / (1 - alpha)`).
- `t`: the step index, for optional diagnostics.

At each new observation `x_t` we update these in O(1) and emit the score.

In [10]:
import numpy as np

def infer(datasets, model_directory_path):
    import os

    bundle = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    cat = bundle["cat"]

    yield  # readiness

    for x_historical, x_online in datasets:
        h_arr = np.asarray(x_historical, dtype=np.float64)
        h_sorted = np.sort(h_arr)
        n_h = len(h_sorted)

        sf_raw = StreamingFeatures(h_arr)
        sf_rank = StreamingFeatures(_rank_history(h_sorted, h_arr))

        peak = 0.0
        for point in x_online:
            pt = float(point)
            f_raw = sf_raw.update(pt)
            rank = int(np.searchsorted(h_sorted, pt, side="right"))
            f_rank = sf_rank.update(_rank_to_gauss(rank, n_h))
            row = np.array([f_raw + f_rank], dtype=np.float64)

            s = float(cat.predict_proba(row, thread_count=1)[0, 1])

            if s > peak: peak = s
            yield 0.3 * peak + 0.7 * s

### Parallelism

If your model is capable of running in parallel, you should try enabling the parallelism mechanism. You can [learn more in the documentation](https://docs.crunchdao.com/competitions/competitions/structural-break-real-time#parallelism).

But the TL;DR is:
- Your model will be run N times, with each call happening in a different process (rather than a thread).

- To avoid concurrency issues, do not write file from the `infer()` function.

- Make sure you don't use too many resources. <br />
  - If your model requires 4 GB of RAM and you want a parallelism of 6, the machine will need at least 24 GB of RAM (+overhead).
  - The same applies to CPUs: try not to use more than the number of CPUs your machine has (+overhead).

- It makes debugging more difficult: if you need to diagnose a bug, revert to a value of `1`.

In [20]:
# @crunch/keep:on
INFER_PARALLELISM = 4

# Uncomment this line to run `infer` with maximum parallelism while leaving one CPU core free for other tasks
# INFER_PARALLELISM = os.cpu_count() - 1

# Uncomment this line to run infer without parallelism
# INFER_PARALLELISM = 1

## Local testing

The Crunch CLI ships with a local tester that reproduces the cloud
environment. <br />
It calls `train()` once, then calls `infer()` with the test data, collects the yielded scores, and writes them to `prediction/prediction.parquet`.

This is the same flow the platform will run.

In [16]:
crunch_tools.test(
    # Uncomment to skip re-training each time
    # force_first_train=False,

    # Uncomment to skip the determinism check
    # no_determinism_check=True,
)

07:19:46 
07:19:46 started
07:19:46 running local test
07:19:46 internet access isn't restricted, no check will be done
07:19:46 
07:19:47 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_test.reduced.parquet (106299 bytes)
data/y_test.reduced.parquet: already exists, file length match
data/y_test_index.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws

07:33:10 
07:33:10 duration - time=00:13:23
07:33:10 memory - before="2.06 GB" after="6.39 GB" consumed="4.33 GB"
07:33:10 Cancelled!


## Previewing the results

In [17]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'prediction/prediction.parquet'

### Computing TS-AUC locally

Below is a straightforward implementation of the TS-AUC metric: group rows by online time step, compute the AUC cross-sectionally at each step (skipping steps that only have one class), and return the weighted average.

This is exactly what the leaderboard uses, just without the private test set.

In [ ]:
# Load the ground-truth labels supplied with the local tester.
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels on (id, time).
merged = prediction.merge(
    y_test,
    how="left",
    left_index=True,
    right_index=True,
)

# Add the online step index (0, 1, 2, ...).
merged["time_online"] = merged.groupby("id").cumcount()

# Weighted per-step AUC.
weighted_auc_sum = 0.0
total_weight     = 0.0

for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue

    auc_t  = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)

    weighted_auc_sum += weight * auc_t
    total_weight     += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

## Ideas for stronger solutions

The baseline above is a starting point, not a competitive submission. <br />
Here are directions that typically pay off, in rough order of effort vs. expected benefit:

**Richer streaming statistics.** The EWMA reacts to *mean* shifts but ignores changes in variance, distributional shape, and serial correlation. Try maintaining, in parallel:

- A **CUSUM** of standardized residuals (cumulative deviation from the historical mean), which accumulates evidence over time rather than decaying it.
- Running **variance** and a ratio against $\sigma_H^2$ to catch volatility shifts.
- Fraction of online points beyond $\pm 2\sigma_H$ or $\pm 3\sigma_H$ (tail-mass change).
- Online **autocorrelation** at a few lags, compared to the historical autocorrelation.
- A **likelihood-ratio CUSUM** under a small autoregressive model fit to the historical segment.

All of these can be updated in O(1) per new observation with a bit of care.

**Combine features with a supervized model:**
- If you collect a handful of incremental features like the ones above into a vector, you have a standard tabular classification problem at every online step.
- A gradient boosting model (LightGBM, XGBoost) trained on `(feature_vector, label_t)` pairs from the training set can learn to weight the features much better than a hand-tuned rule.
- Remember to store *all* the state needed to reproduce the feature vector at inference time in O(1).

**Watch your time budget:**
- With 10,000 series and up to 1,000 online steps each, your code may be asked for up to 10 million scores.
- At any step, anything slower than a few microseconds of Python is a liability. Profile!

# Submitting your notebook

To submit your work, you must:

1. Download your notebook (from Colab, Kaggle, or a local copy).
2. Upload it to the competition platform.
3. Create a **run** to validate it against the full test set.

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [11]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)

warning 101d5543: line 28: column 0: nested import: found 1 nested import in FunctionDef statement
warning a4c0cc77: line 3: column 0: nested import: found 1 nested import in FunctionDef statement


found code file: requirements.txt (190 bytes)


uploading `requirements.txt`:   0%|          | 0.00/190 [00:00<?, ?B/s]

found code file: main.py (15.19 KB)


uploading `main.py`:   0%|          | 0.00/14.8k [00:00<?, ?B/s]

...

uploading `notebook.ipynb`:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

total code size: 55.47 KB
found model file: model.joblib (4 KB)


uploading `model.joblib`:   0%|          | 0.00/3.91k [00:00<?, ?B/s]

total model size: 4 KB
export structural-break-real-time:project/13312/breakyy



---

Next step is to run your submission in the cloud:

### >> https://hub.crunchdao.com/competitions/structural-break-real-time/models/pixelated-anja/breakyy/runs/create?submissionNumber=45

<img alt="Run in the Cloud" src="https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/create-run.gif" height="600px" />
